# Modelling — baseline CNN vs. transfer learning

This notebook is the *narrative*: it states what is being compared and reads the
result. All the machinery — the split, the input pipelines, the metrics — lives in
the `plantvillage` package, so the same objects are used here as in the scripts and
nothing can silently drift between the two.

See `docs/refactoring.md` for what the original notebooks did inline.

## Setup

On Colab, install the package and load the Kaggle credentials from Secrets, so no
key is ever committed.

In [ ]:
# %pip install -q git+https://github.com/<your-user>/plant-disease-classification.git

from plantvillage import DataConfig, Paths, TrainConfig
from plantvillage import data as pv_data
from plantvillage import datasets, evaluation, models, training
from plantvillage.utils import describe_hardware, load_kaggle_credentials, set_seed

set_seed()
load_kaggle_credentials()
print(describe_hardware())

## 1. Data

`load_or_create_split` reads the canonical split if `scripts/prepare_data.py` has
already written it, and otherwise rebuilds it deterministically. Either way
`verify()` asserts the two properties everything downstream relies on: every class
present in every subset, and no file in more than one subset.

In [ ]:
data_config = DataConfig(img_size=128, batch_size=32)
paths = Paths.default().create()

pv_data.download_dataset(paths)
color_dir = pv_data.find_color_dir(paths.raw)
class_names = pv_data.list_class_names(color_dir)
name_to_index = {name: index for index, name in enumerate(class_names)}

split = pv_data.load_or_create_split(color_dir, class_names, paths.splits, data_config)
split

The test set exists from this point on but is **not** requested below — it stays
sealed until `scripts/evaluate.py` opens it, after model selection is finished.

In [ ]:
pipelines = datasets.datasets_from_split(split, name_to_index, data_config)
datasets.assert_raw_pixels(pipelines["train"])   # pixels must still be 0-255 here

y_train = split.indices("train", name_to_index)
y_val = split.indices("val", name_to_index)
class_weight = pv_data.balanced_class_weights(y_train, len(class_names))

print(f"class weights: {min(class_weight.values()):.2f} (most common)"
      f" ... {max(class_weight.values()):.2f} (rarest)")
datasets.preview_batch(pipelines["train"], class_names=class_names);

## 2. Baseline — a small CNN trained from scratch

Deliberately small and un-tuned. Its value is as a reference point, so it has to be
honest rather than impressive. `Rescaling` and the augmentation block are layers
*inside* the model: training, validation and deployment therefore scale images
identically, and the saved model accepts a raw JPEG.

In [ ]:
baseline = models.build_baseline_cnn(len(class_names), data_config.img_size)
baseline.summary()

In [ ]:
train_config = TrainConfig(epochs=20)

baseline_history = training.fit(
    baseline,
    pipelines["train"],
    pipelines["val"],
    epochs=train_config.epochs,
    class_weight=class_weight,
    callbacks=training.default_callbacks(train_config),
    label="baseline CNN",
)

In [ ]:
evaluation.plot_learning_curves(baseline_history, "Baseline CNN");

In [ ]:
baseline_result = evaluation.evaluate_model(
    baseline, pipelines["val"], y_val, class_names, "Baseline CNN (from scratch)"
)
baseline_result.summary()

The distance between accuracy and macro-F1 is the whole story of this dataset: a
model can look strong on accuracy while failing on the rare classes. The per-class
table above is sorted worst-first for exactly that reason.

In [ ]:
evaluation.plot_confusion_matrix(baseline_result);

## 3. Transfer learning — DenseNet-121

Two phases. First the new head learns against a frozen backbone; then the backbone
is unfrozen and fine-tuned at a much lower learning rate. Doing phase 1 first
matters: fine-tuning against a randomly-initialised head would push large, noisy
gradients through the pre-trained weights on the very first batch.

The dashed line on the curves marks where phase 2 begins.

In [ ]:
densenet, backbone = models.build_transfer_model(
    "densenet121", len(class_names), data_config.img_size
)

densenet_history, phase_boundary = training.train_transfer_model(
    densenet,
    backbone,
    pipelines["train"],
    pipelines["val"],
    class_weight=class_weight,
    head_epochs=5,
    finetune_epochs=20,
    finetune_lr=1e-5,
)

In [ ]:
evaluation.plot_learning_curves(densenet_history, "DenseNet-121", phase_boundary);

In [ ]:
densenet_result = evaluation.evaluate_model(
    densenet, pipelines["val"], y_val, class_names, "DenseNet-121 (transfer)"
)
densenet_result.summary()

## 4. Reading the comparison

Both models saw the identical split, the identical pre-processing and the identical
class weights, and were scored by the identical code — so the difference below is
attributable to the architecture and nothing else.

In [ ]:
evaluation.comparison_table([baseline_result, densenet_result])

In [ ]:
delta = evaluation.per_class_delta(
    baseline_result, densenet_result, "baseline", "densenet"
)
print("Largest recall gains:")
print(delta.head(6).to_string(index=False))
print("\nLargest regressions:")
print(delta.tail(6).to_string(index=False))

Aggregate scores move little between two strong models; the interesting changes sit
on the rare and visually similar classes, which is what the per-class delta surfaces.

## 5. Where is the model looking?

Every PlantVillage photo has a plain studio background, so a model can score well by
learning the background rather than the lesion. Grad-CAM is the check on that — and
the reason the lab-to-field caveat belongs in any report on this dataset.

In [ ]:
import numpy as np
from plantvillage import explain

images, labels = next(iter(pipelines["val"]))

for image in images[:3].numpy():
    heatmap = explain.gradcam_heatmap(
        np.expand_dims(image, axis=0), densenet, backbone_name="densenet121"
    )
    explain.overlay_heatmap(image, heatmap)

## 6. Save

Artifacts follow one layout across the project: `artifacts/<run>/` holds the models,
their histories, the metrics and the per-class recalls.

In [ ]:
run_dir = paths.run("notebook_comparison")

baseline.save(run_dir / "baseline_cnn.keras")
densenet.save(run_dir / "densenet121.keras")
training.save_history(baseline_history, run_dir / "baseline_cnn.history.json")
training.save_history(densenet_history, run_dir / "densenet121.history.json")

for result in (baseline_result, densenet_result):
    result.save(run_dir)

evaluation.comparison_table([baseline_result, densenet_result]).to_csv(run_dir / "results.csv")
print("saved to", run_dir)

**Next:** `scripts/evaluate.py --run notebook_comparison` opens the sealed test set
once and reports the validation-to-test gap.